In [ ]:
# ==========================================================
# IMPORTS
# ==========================================================
import os
import sys
import warnings
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from dotenv import load_dotenv

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_mistralai import ChatMistralAI

from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
)

# Ignore certains warnings de compatibilité
warnings.filterwarnings("ignore", category=DeprecationWarning)


# ==========================================================
# CONFIGURATION GÉNÉRALE
# ==========================================================
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

TOKEN = os.getenv("API_KEY", "").strip()
TOKEN_MISTRAL = os.getenv("MISTRAL_API_KEY", "").strip()
TOKEN_OPENAGENDA = os.getenv("OPENAGENDA_API_KEY", "").strip()

API_URL_HEALTH = "http://127.0.0.1:8000/health"
API_URL_REBUILD = "http://127.0.0.1:8000/rebuild"
API_URL_ASK = "http://127.0.0.1:8000/ask"
API_URL_ASK_DEBUG = "http://127.0.0.1:8000/ask/debug"

# Vérifications minimales
if not TOKEN:
    raise ValueError("API_KEY manquante dans le fichier .env")

if not TOKEN_MISTRAL:
    raise ValueError("MISTRAL_API_KEY manquante dans le fichier .env")


# ==========================================================
# FONCTIONS UTILITAIRES
# ==========================================================
def normalize_text(text: str) -> str:
    """
    Normalise une chaîne pour comparaison souple.

    Parameters
    ----------
    text : str
        Texte à normaliser.

    Returns
    -------
    str
        Texte nettoyé.
    """
    if not isinstance(text, str):
        return ""
    return " ".join(text.strip().lower().split())


def safe_json_response(resp: requests.Response) -> dict[str, Any]:
    """
    Convertit une réponse requests en JSON si possible.
    Sinon, retourne un dictionnaire minimal avec le texte brut.

    Parameters
    ----------
    resp : requests.Response
        Réponse HTTP.

    Returns
    -------
    dict[str, Any]
        Données JSON ou fallback.
    """
    try:
        data = resp.json()
        if isinstance(data, dict):
            return data
        return {"raw_json": data}
    except ValueError:
        return {"detail": resp.text}


def ensure_list_of_strings(value: Any) -> list[str]:
    """
    Garantit une liste de chaînes pour les contextes RAGAS.

    Parameters
    ----------
    value : Any
        Valeur brute potentiellement renvoyée par l'API.

    Returns
    -------
    list[str]
        Liste de contextes textuels.
    """
    if value is None:
        return []

    if isinstance(value, list):
        cleaned = []
        for item in value:
            if item is None:
                continue
            if isinstance(item, str):
                text = item.strip()
                if text:
                    cleaned.append(text)
            else:
                text = str(item).strip()
                if text:
                    cleaned.append(text)
        return cleaned

    if isinstance(value, str):
        value = value.strip()
        return [value] if value else []

    text = str(value).strip()
    return [text] if text else []


def ensure_api_ready(zone: str = "Montpellier", scope: str = "city") -> None:
    """
    Vérifie si l'API a déjà un index chargé.
    Si ce n'est pas le cas, déclenche une reconstruction.

    Parameters
    ----------
    zone : str, default="Montpellier"
        Zone à charger si un rebuild est nécessaire.
    scope : str, default="city"
        Scope utilisé pour le rebuild.

    Raises
    ------
    RuntimeError
        Si l'API reste indisponible après tentative de rebuild.
    """
    try:
        resp = requests.get(API_URL_HEALTH, timeout=30)
        resp.raise_for_status()
        data = safe_json_response(resp)

        if data.get("index_loaded", False):
            print("API prête : index déjà chargé.")
            return

        print("Index non chargé. Lancement de /rebuild ...")

        rebuild_resp = requests.post(
            API_URL_REBUILD,
            json={"zone": zone, "scope": scope},
            headers={"x-api-key": TOKEN},
            timeout=300,
        )
        rebuild_resp.raise_for_status()

        rebuild_data = safe_json_response(rebuild_resp)
        print("Rebuild terminé :", rebuild_data)

        final_resp = requests.get(API_URL_HEALTH, timeout=30)
        final_resp.raise_for_status()
        final_data = safe_json_response(final_resp)

        if not final_data.get("index_loaded", False):
            raise RuntimeError("L'index n'est toujours pas chargé après /rebuild.")

        print("API prête après rebuild.")

    except requests.RequestException as exc:
        raise RuntimeError("Impossible de préparer l'API pour l'évaluation.") from exc


def call_ask_api(question: str) -> dict[str, Any]:
    """
    Appelle l'endpoint /ask de l'API RAG.

    Parameters
    ----------
    question : str
        Question utilisateur.

    Returns
    -------
    dict[str, Any]
        Réponse JSON brute de l'API.
    """
    resp = requests.post(
        API_URL_ASK,
        json={"question": question},
        headers={"x-api-key": TOKEN},
        timeout=60,
    )

    print("STATUS /ask :", resp.status_code)

    data = safe_json_response(resp)

    print("ANSWER /ask :")
    print(data.get("answer", ""))

    return data


def run_rag_for_eval(question: str, use_fallback_ask: bool = False) -> dict[str, Any]:
    """
    Appelle l'endpoint /ask/debug afin de récupérer :
    - la réponse générée
    - les contextes récupérés
    - les documents utilisés
    - le nombre de documents
    - une éventuelle erreur

    Si /ask/debug échoue et que use_fallback_ask=True,
    on tente /ask comme solution de secours.

    Parameters
    ----------
    question : str
        Question à poser à l'API.
    use_fallback_ask : bool, default=False
        Active un fallback sur /ask si /ask/debug échoue.

    Returns
    -------
    dict[str, Any]
        Résultat normalisé exploitable pour l'évaluation.
    """
    try:
        resp = requests.post(
            API_URL_ASK_DEBUG,
            json={"question": question},
            headers={"x-api-key": TOKEN},
            timeout=180,
        )

        print(f"QUESTION : {question}")
        print("STATUS /ask/debug :", resp.status_code)

        data = safe_json_response(resp)

        if resp.ok:
            contexts = ensure_list_of_strings(data.get("retrieved_contexts", []))
            documents = data.get("documents", [])
            if not isinstance(documents, list):
                documents = []

            return {
                "question": question,
                "answer": data.get("answer", ""),
                "contexts": contexts,
                "documents": documents,
                "n_docs": data.get("n_docs", len(contexts)),
                "error": None,
                "source_endpoint": "/ask/debug",
            }

        # Si /ask/debug échoue, on renvoie directement l'erreur
        # ou on tente un fallback sur /ask.
        if not use_fallback_ask:
            return {
                "question": question,
                "answer": None,
                "contexts": None,
                "documents": None,
                "n_docs": 0,
                "error": data.get("detail", f"HTTP {resp.status_code}"),
                "source_endpoint": "/ask/debug",
            }

        print("Fallback sur /ask car /ask/debug a échoué.")

        ask_resp = requests.post(
            API_URL_ASK,
            json={"question": question},
            headers={"x-api-key": TOKEN},
            timeout=60,
        )

        print("STATUS /ask :", ask_resp.status_code)

        ask_data = safe_json_response(ask_resp)

        if not ask_resp.ok:
            return {
                "question": question,
                "answer": None,
                "contexts": None,
                "documents": None,
                "n_docs": 0,
                "error": ask_data.get("detail", f"HTTP {ask_resp.status_code}"),
                "source_endpoint": "/ask",
            }

        documents = ask_data.get("documents", [])
        if not isinstance(documents, list):
            documents = []

        return {
            "question": question,
            "answer": ask_data.get("answer", ""),
            "contexts": [],
            "documents": documents,
            "n_docs": ask_data.get("n_docs", 0),
            "error": "ask_debug_failed_fallback_to_ask",
            "source_endpoint": "/ask",
        }

    except requests.exceptions.RequestException as exc:
        return {
            "question": question,
            "answer": None,
            "contexts": None,
            "documents": None,
            "n_docs": 0,
            "error": str(exc),
            "source_endpoint": "request_exception",
        }


# ==========================================================
# PRÉPARATION DE L'API
# ==========================================================
ensure_api_ready(zone="Montpellier", scope="city")


# ==========================================================
# JEU D'ÉVALUATION
# Version RAGAS-friendly
# ==========================================================
eval_questions_positive = [
    {
        "question": "Y a-t-il des expositions à Montpellier ?",
        "ground_truth": (
            "Liste d'événements pour Y a-t-il des expositions à Montpellier ?:\n"
            "- L'association Miss'terre fait son expo d'été ! "
            "(date : 2025/06/27 au 2025/06/28, lieu : Association Miss'terre).\n"
            "- Exposition « Terre de Bâtisseuses » "
            "(date : 2025/06/27 au 2025/07/24, lieu : CAUE de l'Hérault).\n"
            "- Une musique de cirque ? "
            "(date : 2026/01/27, lieu : Galerie ATRIUM)."
        ),
        "group": "positive",
    },
    {
        "question": "Quels événements ont lieu à Montpellier en septembre 2025 ?",
        "ground_truth": (
            "Liste d'événements pour Quels événements ont lieu à Montpellier en septembre 2025 ?:\n"
            "- Concert : ZOHUD "
            "(date : 2025/09/21, lieu : Domaine D’O).\n"
            "- Contes : JIHAD DARWICHE "
            "(date : 2025/09/20 au 2025/09/21, lieu : Domaine d'O).\n"
            "- Concert : Le DUO SABIL & VINCENT SEGAL "
            "(date : 2025/09/13, lieu : Domaine d'O)."
        ),
        "group": "positive",
    },
    {
        "question": "Quels événements ont lieu le 20 septembre 2025 à Montpellier ?",
        "ground_truth": (
            "Liste d'événements pour Quels événements ont lieu le 20 septembre 2025 à Montpellier ?:\n"
            "- Contes : JIHAD DARWICHE "
            "(date : 2025/09/20 au 2025/09/21, lieu : Domaine d'O)."
        ),
        "group": "positive",
    },
    {
        "question": "Quels événements ont lieu le week-end du 20 au 21 septembre 2025 à Montpellier ?",
        "ground_truth": (
            "Liste d'événements pour Quels événements ont lieu le week-end du 20 au 21 septembre 2025 à Montpellier ?:\n"
            "- Concert : ZOHUD "
            "(date : 2025/09/21, lieu : Domaine D’O).\n"
            "- Contes : JIHAD DARWICHE "
            "(date : 2025/09/20 au 2025/09/21, lieu : Domaine d'O).\n"
            "- Chez le Poët Jeannot "
            "(date : 2025/09/21, lieu : Chez Jeannot)."
        ),
        "group": "positive",
    },
]

eval_questions_ambiguous = [
    {
        "question": "Quels événements culturels ont lieu à Montpellier ?",
        "ground_truth": (
            "Liste d'événements pour Quels événements culturels ont lieu à Montpellier ?:\n"
            "- Critical Mass Montpellier "
            "(date : 2025/05/30, lieu : Place de la Comédie).\n"
            "- 2030 Micro-Festival #1 "
            "(date : 2025/05/25, lieu : Maison Pour tous Georges Brassens).\n"
            "- Une musique de cirque ? "
            "(date : 2026/01/27, lieu : Galerie ATRIUM)."
        ),
        "group": "ambiguous",
    },
    {
        "question": "Quels événements ont lieu à Montpellier le week-end du 28 au 29 mars 2026 ?",
        "ground_truth": (
            "Liste d'événements pour Quels événements ont lieu à Montpellier le week-end du 28 au 29 mars 2026 ?:\n"
            "- 12° Vinyl Pop-Up de Montpellier "
            "(date : 2026/03/29, lieu : La Halle tropisme)."
        ),
        "group": "ambiguous",
    },
    {
        "question": "Quels événements gratuits ont lieu à Montpellier ?",
        "ground_truth": (
            "Liste d'événements pour Quels événements gratuits ont lieu à Montpellier ?:\n"
            "- GRAND DÉSTOCKAGE - Braderie à Montpellier "
            "(date : 2025/12/06, lieu : Marché gare).\n"
            "- 12° Vinyl Pop-Up de Montpellier "
            "(date : 2026/03/29, lieu : La Halle tropisme).\n"
            "- Je participe aux ateliers de réparations collectives "
            "(date : 2025/10/18, lieu : REPAIR CAFÉ MONTPELLIER)."
        ),
        "group": "ambiguous",
    },
    {
        "question": "Y a-t-il des concerts de rock à Montpellier ?",
        "ground_truth": (
            "Liste d'événements pour Y a-t-il des concerts de rock à Montpellier ?:\n"
            "- EX TENEBRIS LUX - GRAVEKVLT + ZOLDIER NOIZ + AREIS - 18 OCT. - ANTIROUILLE - MONTPELLIER "
            "(date : 2025/10/18, lieu : L'Antirouille).\n"
            "- Mr Jack en concert au Gazette Café "
            "(date : 2026/03/14, lieu : Gazette Café).\n"
            "- Concert : ZOHUD "
            "(date : 2025/09/21, lieu : Domaine D’O)."
        ),
        "group": "ambiguous",
    },
]
eval_questions_negative = [
    {
        "question": "Quels événements ont lieu à Paris ?",
        "ground_truth": (
            "Je n'ai trouvé aucun événement correspondant dans les données disponibles."
        ),
        "group": "negative",
    },
]

eval_questions = (
    eval_questions_positive
    + eval_questions_ambiguous
    + eval_questions_negative
)

print("Nombre total de questions d'évaluation :", len(eval_questions))

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)


# ==========================================================
# COLLECTE DES RÉSULTATS DE L'API
# ==========================================================
rows = []

for item in eval_questions:
    result = run_rag_for_eval(
        question=item["question"],
        use_fallback_ask=False,   # Passe à True si tu veux un secours temporaire
    )

    rows.append(
        {
            "question": item["question"],
            "answer": result["answer"],
            "contexts": result["contexts"],
            "documents": result["documents"],
            "n_docs": result["n_docs"],
            "ground_truth": item["ground_truth"],
            "group": item["group"],
            "error": result["error"],
            "source_endpoint": result.get("source_endpoint"),
        }
    )

rows_df = pd.DataFrame(rows)

print("\nAperçu des résultats bruts :")
display(rows_df)

# Vérification rapide des erreurs API
error_rows = rows_df[rows_df["error"].notna()].copy()
print("Nombre d'erreurs :", len(error_rows))

if not error_rows.empty:
    display(error_rows[["question", "error", "source_endpoint"]])


# ==========================================================
# CONSTRUCTION DU DATASET RAGAS
# ==========================================================
ragas_rows = [
    {
        "user_input": row["question"],
        "response": row["answer"],
        "retrieved_contexts": ensure_list_of_strings(row["contexts"]),
        "reference": row["ground_truth"],
        "group": row["group"],
    }
    for row in rows
    if row["answer"] is not None
]

print("Nombre de lignes exploitables pour RAGAS :", len(ragas_rows))

if not ragas_rows:
    raise ValueError("Aucune ligne exploitable pour RAGAS.")

print("\nExemple de ligne RAGAS :")
print(ragas_rows[0])


# ==========================================================
# CONFIGURATION DES ÉVALUATEURS
# ==========================================================
llm = ChatMistralAI(
    model="mistral-small-latest",
    api_key=TOKEN_MISTRAL,
    temperature=0,
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)


# ==========================================================
# SPLIT DU DATASET PAR GROUPE
# ==========================================================
rows_positive = [row for row in ragas_rows if row["group"] == "positive"]
rows_ambiguous = [row for row in ragas_rows if row["group"] == "ambiguous"]
rows_negative = [row for row in ragas_rows if row["group"] == "negative"]

print("\nNombre de questions positives :", len(rows_positive))
print("Nombre de questions ambiguës :", len(rows_ambiguous))
print("Nombre de questions négatives :", len(rows_negative))


# ==========================================================
# ÉVALUATION RAGAS - QUESTIONS POSITIVES
# ==========================================================
if rows_positive:
    dataset_positive = EvaluationDataset.from_list(rows_positive)

    result_positive = evaluate(
        dataset=dataset_positive,
        metrics=[
            Faithfulness(),
            AnswerRelevancy(strictness=1),
            ContextPrecision(),
            ContextRecall(),
        ],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        raise_exceptions=False,
    )

    df_positive = result_positive.to_pandas().reset_index(drop=True)
    df_positive["group_eval"] = "positive"
    df_positive["question"] = [row["user_input"] for row in rows_positive]

    print("\nRésultats RAGAS - Positives :")
    display(df_positive)
else:
    df_positive = pd.DataFrame()
    print("\nAucune question positive exploitable.")


# ==========================================================
# ÉVALUATION RAGAS - QUESTIONS AMBIGUËS
# ==========================================================
if rows_ambiguous:
    dataset_ambiguous = EvaluationDataset.from_list(rows_ambiguous)

    result_ambiguous = evaluate(
        dataset=dataset_ambiguous,
        metrics=[
            Faithfulness(),
            AnswerRelevancy(strictness=1),
            ContextPrecision(),
            ContextRecall(),
        ],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        raise_exceptions=False,
    )

    df_ambiguous = result_ambiguous.to_pandas().reset_index(drop=True)
    df_ambiguous["group_eval"] = "ambiguous"
    df_ambiguous["question"] = [row["user_input"] for row in rows_ambiguous]

    print("\nRésultats RAGAS - Ambiguës :")
    display(df_ambiguous)
else:
    df_ambiguous = pd.DataFrame()
    print("\nAucune question ambiguë exploitable.")


# ==========================================================
# ÉVALUATION MÉTIER - QUESTIONS NÉGATIVES
# ==========================================================
df_negative_business = rows_df[rows_df["group"] == "negative"].copy()

if not df_negative_business.empty:
    df_negative_business["negative_exact_match"] = df_negative_business.apply(
        lambda row: int(
            normalize_text(row["answer"]) == normalize_text(row["ground_truth"])
        ),
        axis=1,
    )

    negative_score = df_negative_business["negative_exact_match"].mean()
else:
    negative_score = None
    df_negative_business["negative_exact_match"] = []

print("\nScore métier négatif :", negative_score)
display(
    df_negative_business[
        ["question", "answer", "ground_truth", "negative_exact_match"]
    ]
)


# ==========================================================
# TABLEAUX RÉCAPITULATIFS
# ==========================================================
positive_mean = (
    df_positive[
        ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
    ].mean()
    if not df_positive.empty
    else pd.Series(dtype=float)
)

ambiguous_mean = (
    df_ambiguous[
        ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
    ].mean()
    if not df_ambiguous.empty
    else pd.Series(dtype=float)
)

results_summary = {
    "positive_mean": positive_mean,
    "ambiguous_mean": ambiguous_mean,
    "negative_exact_match_mean": negative_score,
}

print("\nRésumé des scores :")
for key, value in results_summary.items():
    print(f"\n{key}")
    print(value)


# ==========================================================
# DATAFRAME DE SYNTHÈSE
# ==========================================================
summary_df = pd.DataFrame({
    "group": ["positive", "ambiguous", "negative"],
    "faithfulness": [
        df_positive["faithfulness"].mean() if not df_positive.empty else None,
        df_ambiguous["faithfulness"].mean() if not df_ambiguous.empty else None,
        None,
    ],
    "answer_relevancy": [
        df_positive["answer_relevancy"].mean() if not df_positive.empty else None,
        df_ambiguous["answer_relevancy"].mean() if not df_ambiguous.empty else None,
        None,
    ],
    "context_precision": [
        df_positive["context_precision"].mean() if not df_positive.empty else None,
        df_ambiguous["context_precision"].mean() if not df_ambiguous.empty else None,
        None,
    ],
    "context_recall": [
        df_positive["context_recall"].mean() if not df_positive.empty else None,
        df_ambiguous["context_recall"].mean() if not df_ambiguous.empty else None,
        None,
    ],
    "negative_exact_match": [
        None,
        None,
        negative_score,
    ],
})

print("\nTableau de synthèse final :")
display(summary_df)


# ==========================================================
# BONUS : TABLEAUX DÉTAILLÉS AVEC QUESTION + SCORE
# ==========================================================
if not df_positive.empty:
    positive_detailed = pd.DataFrame({
        "question": [row["user_input"] for row in rows_positive],
        "faithfulness": df_positive["faithfulness"],
        "answer_relevancy": df_positive["answer_relevancy"],
        "context_precision": df_positive["context_precision"],
        "context_recall": df_positive["context_recall"],
    })

    print("\nDétail des scores - Positives :")
    display(positive_detailed)
else:
    print("\nPas de détail positif disponible.")


if not df_ambiguous.empty:
    ambiguous_detailed = pd.DataFrame({
        "question": [row["user_input"] for row in rows_ambiguous],
        "faithfulness": df_ambiguous["faithfulness"],
        "answer_relevancy": df_ambiguous["answer_relevancy"],
        "context_precision": df_ambiguous["context_precision"],
        "context_recall": df_ambiguous["context_recall"],
    })

    print("\nDétail des scores - Ambiguës :")
    display(ambiguous_detailed)
else:
    print("\nPas de détail ambigu disponible.")

API prête : index déjà chargé.
Nombre total de questions d'évaluation : 9
QUESTION : Y a-t-il des expositions à Montpellier ?
STATUS /ask/debug : 200
QUESTION : Quels événements ont lieu à Montpellier en septembre 2025 ?
STATUS /ask/debug : 200
QUESTION : Quels événements ont lieu le 20 septembre 2025 à Montpellier ?
STATUS /ask/debug : 500
QUESTION : Quels événements ont lieu le week-end du 20 au 21 septembre 2025 à Montpellier ?
STATUS /ask/debug : 500
QUESTION : Quels événements culturels ont lieu à Montpellier ?
STATUS /ask/debug : 500
QUESTION : Quels événements ont lieu à Montpellier le week-end du 28 au 29 mars 2026 ?
STATUS /ask/debug : 500
QUESTION : Quels événements gratuits ont lieu à Montpellier ?
STATUS /ask/debug : 500
QUESTION : Y a-t-il des concerts de rock à Montpellier ?
STATUS /ask/debug : 500
QUESTION : Quels événements ont lieu à Paris ?
STATUS /ask/debug : 500

Aperçu des résultats bruts :


,question,answer,contexts,documents,n_docs,ground_truth,group,error,source_endpoint
0,Y a-t-il des expositions à Montpellier ?,"Liste d'événements pour Y a-t-il des expositions à Montpellier ? :\n- L'association Miss'terre fait son expo d'été ! (date : 2025/06/27 au 2025/06/28, lieu : Association Miss'terre)\n- Exposition « Terre de Bâtisseuses » (date : 2025/06/27 au 2025/07/24, lieu : CAUE de l'Hérault)\n- Une musique de cirque ? (date : 2026/01/27, lieu : Galerie ATRIUM)","[Événement 1\nTitre : L'association Miss'terre fait son expo d'été !\nLieu : Association Miss'terre\nVille : Montpellier\nRégion : Occitanie\nDate de début : 2025-06-27\nDate de fin : 2025-06-28\nType : exposition\nGenre musical : \nTarification : gratuit\nDescription : Exposition\nURL : https://openagenda.com/contributeurs-petit-agenda/events/lassociation-missterre-fait-son-expo-dete, Événement 2\nTitre : Exposition « Terre de Bâtisseuses »\nLieu : CAUE de l'Hérault\nVille : Montpellier\nRégion : Occitanie\nDate de début : 2025-06-27\nDate de fin : 2025-07-24\nType : exposition\nGenre musical : \nTarification : gratuit\nDescription : Pour ce début d'été, la MAOM vous invite à venir explorer « Terre de Bâtisseuses », une exposition signée Alizée Cugney et Claire Dycha, co-gérantes de l’association éponyme.\nURL : https://openagenda.com/reseaudesma/events/exposition-terre-de-batisseuses, Événement 3\nTitre : Une musique de cirque ?\nLieu : Galerie ATRIUM\nVille : Montpellier\nRégion : Occitanie\nDate de début : 2026-01-27\nDate de fin : 2026-01-27\nType : exposition\nGenre musical : \nTarification : gratuit\nDescription : Soirée d'inauguration AUDITORIUM de l'ATRIUM\nURL : https://openagenda.com/theatre-de-la-palabre/events/soiree-dinauguration-auditorium-de-latrium]","[{'title': 'L'association Miss'terre fait son expo d'été !', 'location_name': 'Association Miss'terre', 'city': 'Montpellier', 'region': 'Occitanie', 'first_date': '2025-06-27', 'last_date': '2025-06-28', 'event_type': 'exposition', 'url': 'https://openagenda.com/contributeurs-petit-agenda/events/lassociation-missterre-fait-son-expo-dete', 'price_info': 'gratuit', 'is_free': True, 'keywords_title': ['association', 'ete', 'expo', 'fait', 'miss', 'son', 'terre'], 'score': 26.377842116355897}, {'title': 'Exposition « Terre de Bâtisseuses »', 'location_name': 'CAUE de l'Hérault', 'city': 'Montpellier', 'region': 'Occitanie', 'first_date': '2025-06-27', 'last_date': '2025-07-24', 'event_type': 'exposition', 'url': 'https://openagenda.com/reseaudesma/events/exposition-terre-de-batisseuses', 'price_info': 'gratuit', 'is_free': True, 'keywords_title': ['batisseuses', 'exposition', 'terre'], 'score': 26.35667221546173}, {'title': 'Une musique de cirque ?', 'location_name': 'Galerie ATRIUM', 'city': 'Montpellier', 'region': 'Occitanie', 'first_date': '2026-01-27', 'last_date': '2026-01-27', 'event_type': 'exposition', 'url': 'https://openagenda.com/theatre-de-la-palabre/events/soiree-dinauguration-auditorium-de-latrium', 'price_info': 'gratuit', 'is_free': True, 'keywords_title': ['cirque', 'musique'], 'score': 26.33141508102417}]",3,"Liste d'événements pour Y a-t-il des expositions à Montpellier ?:\n- L'association Miss'terre fait son expo d'été ! (date : 2025/06/27 au 2025/06/28, lieu : Association Miss'terre).\n- Exposition « Terre de Bâtisseuses » (date : 2025/06/27 au 2025/07/24, lieu : CAUE de l'Hérault).\n- Une musique de cirque ? (date : 2026/01/27, lieu : Galerie ATRIUM).",positive,None,/ask/debug
1,Quels événements ont lieu à Montpellier en septembre 2025 ?,"Liste d'événements pour Quels événements ont lieu à Montpellier en septembre 2025 ? :\n- Concert : ZOHUD (date : 2025/09/21, lieu : Domaine D’O)\n- Contes : JIHAD DARWICHE (date : 2025/09/20 au 2025/09/21, lieu : Domaine d'O)\n- Concert : Le DUO SABIL & VINCENT SEGAL (date : 2025/09/13, lieu : Domaine d'O)","[Événement 1\nTitre : Concert : ZOHUD\nLieu : Domaine D’O\nVille : Montpellier\nRégion : Occitanie\nDate de début : 2025-09-21\nDat

Nombre d'erreurs : 7


,question,error,source_endpoint
2,Quels événements ont lieu le 20 septembre 2025 à Montpellier ?,Erreur interne du serveur.,/ask/debug
3,Quels événements ont lieu le week-end du 20 au 21 septembre 2025 à Montpellier ?,Erreur interne du serveur.,/ask/debug
4,Quels événements culturels ont lieu à Montpellier ?,Erreur interne du serveur.,/ask/debug
5,Quels événements ont lieu à Montpellier le week-end du 28 au 29 mars 2026 ?,Erreur interne du serveur.,/ask/debug
6,Quels événements gratuits ont lieu à Montpellier ?,Erreur interne du serveur.,/ask/debug
7,Y a-t-il des concerts de rock à Montpellier ?,Erreur interne du serveur.,/ask/debug
8,Quels événements ont lieu à Paris ?,Erreur interne du serveur.,/ask/debug


Nombre de lignes exploitables pour RAGAS : 2

Exemple de ligne RAGAS :
{'user_input': 'Y a-t-il des expositions à Montpellier ?', 'response': "Liste d'événements pour Y a-t-il des expositions à Montpellier ? :\n- L'association Miss'terre fait son expo d'été ! (date : 2025/06/27 au 2025/06/28, lieu : Association Miss'terre)\n- Exposition « Terre de Bâtisseuses » (date : 2025/06/27 au 2025/07/24, lieu : CAUE de l'Hérault)\n- Une musique de cirque ? (date : 2026/01/27, lieu : Galerie ATRIUM)", 'retrieved_contexts': ["Événement 1\nTitre : L'association Miss'terre fait son expo d'été !\nLieu : Association Miss'terre\nVille : Montpellier\nRégion : Occitanie\nDate de début : 2025-06-27\nDate de fin : 2025-06-28\nType : exposition\nGenre musical : \nTarification : gratuit\nDescription : Exposition\nURL : https://openagenda.com/contributeurs-petit-agenda/events/lassociation-missterre-fait-son-expo-dete", "Événement 2\nTitre : Exposition « Terre de Bâtisseuses »\nLieu : CAUE de l'Hérault\nVille 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Nombre de questions positives : 2
Nombre de questions ambiguës : 0
Nombre de questions négatives : 0


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]